<a href="https://colab.research.google.com/github/omargarawani/Sign-Language-Fingerspelling-Assistant-ASL-Alphabet-/blob/main/Lstm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import json
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

In [ ]:
with open('splits.json', 'r', encoding='utf-8') as f:
    splits = json.load(f)

train_pairs = splits['train']   # [[english, arabic], ...]
val_pairs   = splits['val']
test_pairs  = splits['test']

all_pairs     = train_pairs + val_pairs + test_pairs
english_words = [p[0].lower().strip() for p in all_pairs]
arabic_words  = [p[1] for p in all_pairs]

print(f"Train: {len(train_pairs)} | Val: {len(val_pairs)} | Test: {len(test_pairs)}")
print("Sample:", train_pairs[:3])

Train: 4000 | Val: 500 | Test: 500
Sample: [['kareemsami', 'كاريمسامي'], ['stephanieing', 'ستيفانياينج'], ['henryity', 'هينريتي']]


In [ ]:
PAD   = '<PAD>'
START = '<START>'
END   = '<END>'

def build_vocab(words):
    chars = sorted(set(ch for w in words for ch in w))
    vocab = [PAD, START, END] + chars
    w2i   = {c: i for i, c in enumerate(vocab)}
    i2w   = {i: c for c, i in w2i.items()}
    return vocab, w2i, i2w

eng_vocab, eng_w2i, eng_i2w = build_vocab(english_words)
ara_vocab, ara_w2i, ara_i2w = build_vocab(arabic_words)

ENG_VOCAB_SIZE = len(eng_vocab)
ARA_VOCAB_SIZE = len(ara_vocab)
ENC_MAX_LEN    = max(len(w) for w in english_words)
DEC_MAX_LEN    = max(len(w) for w in arabic_words) + 2

print(f"English vocab: {ENG_VOCAB_SIZE} | max len: {ENC_MAX_LEN}")
print(f"Arabic  vocab: {ARA_VOCAB_SIZE} | max len: {DEC_MAX_LEN}")

English vocab: 28 | max len: 15
Arabic  vocab: 23 | max len: 16


In [ ]:
def encode_eng(word):
    ids = [eng_w2i.get(ch, 0) for ch in word]
    ids += [eng_w2i[PAD]] * (ENC_MAX_LEN - len(ids))
    return ids

def encode_ara(word):
    ids = [ara_w2i[START]] + [ara_w2i.get(ch, 0) for ch in word] + [ara_w2i[END]]
    ids += [ara_w2i[PAD]] * (DEC_MAX_LEN - len(ids))
    return ids

encoder_inputs     = np.array([encode_eng(w) for w in english_words])
dec_full           = np.array([encode_ara(w) for w in arabic_words])
decoder_inputs     = dec_full[:, :-1]
decoder_targets    = dec_full[:, 1:]
decoder_targets_oh = tf.keras.utils.to_categorical(decoder_targets, num_classes=ARA_VOCAB_SIZE)

print("encoder_inputs :", encoder_inputs.shape)
print("decoder_inputs :", decoder_inputs.shape)
print("decoder_targets:", decoder_targets_oh.shape)

encoder_inputs : (5000, 15)
decoder_inputs : (5000, 15)
decoder_targets: (5000, 15, 23)


In [ ]:
EMBED_DIM  = 64
LATENT_DIM = 256

# Encoder
enc_input  = Input(shape=(ENC_MAX_LEN,), name='enc_input')
enc_embed  = Embedding(ENG_VOCAB_SIZE, EMBED_DIM, mask_zero=True, name='enc_embedding')(enc_input)
_, state_h, state_c = LSTM(LATENT_DIM, return_state=True, name='enc_lstm')(enc_embed)
enc_states = [state_h, state_c]

# Decoder
dec_input  = Input(shape=(DEC_MAX_LEN - 1,), name='dec_input')
dec_embed  = Embedding(ARA_VOCAB_SIZE, EMBED_DIM, mask_zero=True, name='dec_embedding')(dec_input)
dec_lstm   = LSTM(LATENT_DIM, return_sequences=True, return_state=True, name='dec_lstm')
dec_out, _, _ = dec_lstm(dec_embed, initial_state=enc_states)
dec_dense  = Dense(ARA_VOCAB_SIZE, activation='softmax', name='dec_output')(dec_out)

model = Model([enc_input, dec_input], dec_dense)
model.compile(optimizer=Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ enc_input           │ (None, 15)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dec_input           │ (None, 15)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_embedding       │ (None, 15, 64)    │      1,792 │ enc_input[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 15)        │          0 │ enc_input[0][0]   │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dec_embedding       │ (None, 15, 64)    │      1,472 │ dec_input[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_lstm (LSTM)     │ [(None, 256),     │    328,704 │ enc_embedding[0]… │
│                     │ (None, 256),      │            │ not_equal[0][0]   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dec_lstm (LSTM)     │ [(None, 15, 256), │    328,704 │ dec_embedding[0]… │
│                     │ (None, 256),      │            │ enc_lstm[0][1],   │
│                     │ (None, 256)]      │            │ enc_lstm[0][2]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dec_output (Dense)  │ (None, 15, 23)    │      5,911 │ dec_lstm[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 666,583 (2.54 MB)

 Trainable params: 666,583 (2.54 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
#Train

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
checkpoint = ModelCheckpoint('lstm_seq2seq.h5', monitor='val_loss', save_best_only=True)

history = model.fit(
    [encoder_inputs, decoder_inputs],
    decoder_targets_oh,
    batch_size=64,
    epochs=100,
    validation_split=0.1,
    callbacks=[early_stop, checkpoint]
)

model.save('lstm_seq2seq.h5')
print('Saved → lstm_seq2seq.h5')

Epoch 1/100
70/71 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.2657 - loss: 2.5617

71/71 ━━━━━━━━━━━━━━━━━━━━ 8s 19ms/step - accuracy: 0.3248 - loss: 2.2685 - val_accuracy: 0.3643 - val_loss: 2.0254
Epoch 2/100
66/71 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.3910 - loss: 1.9507

71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.4175 - loss: 1.8728 - val_accuracy: 0.4533 - val_loss: 1.7336
Epoch 3/100
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.4722 - loss: 1.6881

71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.4887 - loss: 1.6394 - val_accuracy: 0.5085 - val_loss: 1.5647
Epoch 4/100
66/71 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5286 - loss: 1.5109

71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.5472 - loss: 1.4590 - val_accuracy: 0.5771 - val_loss: 1.3690
Epoch 5/100
66/71 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6020 - loss: 1.3036

...

71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 1.0000 - loss: 6.5248e-04 - val_accuracy: 0.9889 - val_loss: 0.0407
Epoch 64/100
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 1.0000 - loss: 6.0994e-04

71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 1.0000 - loss: 6.2266e-04 - val_accuracy: 0.9892 - val_loss: 0.0400
Epoch 65/100
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 1.0000 - loss: 5.9404e-04 - val_accuracy: 0.9889 - val_loss: 0.0405
Epoch 66/100
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 1.0000 - loss: 5.6616e-04 - val_accuracy: 0.9885 - val_loss: 0.0406
Epoch 67/100
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 1.0000 - loss: 5.3790e-04 - val_accuracy: 0.9887 - val_loss: 0.0408
Epoch 68/100
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 1.0000 - loss: 5.1171e-04 - val_accuracy: 0.9891 - val_loss: 0.0407
Epoch 69/100
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 1.0000 - loss: 4.9234e-04 - val_accuracy: 0.9885 - val_loss: 0.0410
Epoch 70/100
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 1.0000 - loss: 4.6885e-04 - val_accuracy: 0.9887 - val_loss: 0.0409
Epoch 71/100
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 1.0000 - loss: 4.

71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 1.0000 - loss: 4.4698e-04 - val_accuracy: 0.9898 - val_loss: 0.0400
Epoch 72/100
71/71 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 1.0000 - loss: 4.2819e-04 - val_accuracy: 0.9885 - val_loss: 0.0407
Epoch 73/100
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 1.0000 - loss: 4.0785e-04 - val_accuracy: 0.9891 - val_loss: 0.0407
Epoch 74/100
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 1.0000 - loss: 3.8952e-04 - val_accuracy: 0.9881 - val_loss: 0.0409
Epoch 75/100
69/71 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 1.0000 - loss: 3.6836e-04

71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 1.0000 - loss: 3.7340e-04 - val_accuracy: 0.9900 - val_loss: 0.0395
Epoch 76/100
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 1.0000 - loss: 3.5675e-04 - val_accuracy: 0.9898 - val_loss: 0.0403
Epoch 77/100
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 1.0000 - loss: 3.4026e-04 - val_accuracy: 0.9892 - val_loss: 0.0401
Epoch 78/100
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 1.0000 - loss: 3.2550e-04 - val_accuracy: 0.9892 - val_loss: 0.0400
Epoch 79/100
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 1.0000 - loss: 3.1259e-04 - val_accuracy: 0.9898 - val_loss: 0.0400
Epoch 80/100
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 1.0000 - loss: 2.9850e-04 - val_accuracy: 0.9892 - val_loss: 0.0399
Epoch 81/100
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 1.0000 - loss: 2.8575e-04 - val_accuracy: 0.9898 - val_loss: 0.0400
Epoch 82/100
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 1.0000 - loss: 2.

Saved → lstm_seq2seq.h5


In [ ]:
# Inference models

# Encoder: English input → hidden states
inf_enc_model = Model(enc_input, enc_states)

# Decoder: 1 char + states → next char + new states
inf_h_in   = Input(shape=(LATENT_DIM,), name='inf_h')
inf_c_in   = Input(shape=(LATENT_DIM,), name='inf_c')
inf_dec_in = Input(shape=(1,),          name='inf_dec_in')

inf_embed        = model.get_layer('dec_embedding')(inf_dec_in)
inf_out, inf_h, inf_c = model.get_layer('dec_lstm')(inf_embed, initial_state=[inf_h_in, inf_c_in])
inf_dense_out    = model.get_layer('dec_output')(inf_out)

inf_dec_model = Model(
    [inf_dec_in, inf_h_in, inf_c_in],
    [inf_dense_out, inf_h, inf_c]
)

In [ ]:
def transliterate(english_word):
    """
    Takes an English word and returns its Arabic transliteration.
    Used by the main pipeline after the CNN predicts letters.

    Example:
        transliterate('ahmed')    → 'احمد'
        transliterate('computer') → 'كمبيوتر'
    """
    word = english_word.lower().strip()

    ids = [eng_w2i.get(ch, 0) for ch in word]
    ids += [eng_w2i[PAD]] * (ENC_MAX_LEN - len(ids))
    enc_in = np.array([ids])

    states = inf_enc_model.predict(enc_in, verbose=0)

    target = np.array([[ara_w2i[START]]])
    result = []

    for _ in range(30):
        out, h, c = inf_dec_model.predict([target] + states, verbose=0)
        pred_id   = np.argmax(out[0, 0, :])
        pred_char = ara_i2w[pred_id]

        if pred_char in (END, PAD):
            break

        result.append(pred_char)
        target = np.array([[pred_id]])
        states = [h, c]

    return ''.join(result)


# ── Test on test split ────────────────────────────────────────────────────────
print(f"{'English':20s} {'Predicted':20s} {'Ground Truth'}")
print('-' * 60)
for eng, ara_gt in test_pairs[:10]:
    pred = transliterate(eng)
    print(f"{eng:20s} {pred:20s} {ara_gt}")

English              Predicted            Ground Truth
------------------------------------------------------------
joeless              جويليس               جويليس
virginiable          فيرجينيابلي          فيرجينيابلي
marieed              ماريدي               ماريد
teresaawi            تيريساوي             تيريساوي
emilyate             يميلياتي             يميلياتي
tylerson             تيليرسون             تيليرسون
eugenean             يوجينيان             يوجينيان
thomasson            ثوماسون              ثوماسون
joshuaive            جوشوايفي             جوشوايفي
henryling            هينريلينج            هينريلينج
